# Comma Compression — Pruning hypotheses (A100/H100)

**Goal:** shave ≥3 KB off `gen_3090.pt.e80.ckpt` (currently 54078 model_bytes → score 0.2921 with sidecar). Every 1 KB saved = -0.000667 on score. 3 KB → 0.2901; 9 KB → 0.2861.

**Key insight from local analysis:**
- `pose_mlp` (3× nn.Linear) eats **13.7 KB / 25%** of model bytes because it's stored as FP16 (everything else is FP4).
- Two single layers (`pose_mlp.2`, `pose_mlp.4`) are **6.5 KB each** — the biggest weights in the entire model.
- Naive FP4 round-trip of pose_mlp blows up score (1.05). **QAT recovery is required.**

**Hypotheses (ranked by ROI):**
| # | Method | Bytes saved | Risk | Suggested time |
|---|---|---|---|---|
| **H1** | pose_mlp → QLinear (FP4) + QAT | ~9.6 KB | med | **45 min** ← START HERE |
| H2 | pose_mlp shrink: 64→32 hidden + QAT | ~6 KB | low-med | 45 min |
| H3 | SVD-low-rank pose_mlp.2/4 + QAT | ~4 KB | low | 30 min |
| H4 | drop h1.r2 (2nd FiLMRes) + QAT | ~7 KB | high | 60 min |
| H5 | trunk C2 64→48 + QAT | ~3 KB | high | 60 min |
| H6 | shrink HEAD_HIDDEN 52→32 + QAT | ~1 KB | low | 30 min |
| H7 | drop h2.r1 (Res in Head2) + QAT | ~3 KB | med | 45 min |
| **H8** | **H1 + H4 combined** | **~16 KB** 🎯 | high | 90 min |
| H9 | distill smaller student (C1 56→44) | ~6 KB | high | 90 min |

**Compute plan (6h A100):** Run H1 first (validated big win), then H8 (combined), then H3 (low-risk addition). Rest are stretch.

**Frequent eval policy:** every 5 epochs + every 5 min wallclock. ALL ckpts saved to Drive (don't overwrite — last time intermediate beat final).

## 0. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Want A100 or H100. L4/T4 will be slow. Change via Runtime → Change runtime type.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/comma_compression_runs')
PRUNE_DIR  = DRIVE_ROOT / 'prune'
PRUNE_DIR.mkdir(parents=True, exist_ok=True)
print(f'All ckpts/logs saved under: {PRUNE_DIR}')

## 2. Setup repo + deps

In [ ]:
%cd /content
REPO_URL = 'https://github.com/BradyMeighan/comma_video_compression_challenge.git'
BRANCH   = 'autoresearch/apr29'
!if [ ! -d comma_video_compression_challenge ]; then git clone --branch $BRANCH $REPO_URL; fi
%cd /content/comma_video_compression_challenge
!git fetch origin $BRANCH && git checkout $BRANCH && git pull origin $BRANCH
!pip -q install einops safetensors brotli zstandard segmentation_models_pytorch timm tqdm av

## 3. Stage e80 checkpoint + (optionally) data cache from Drive

If `full_split_600all.pt` (~3.6 GB) is on your Drive at `comma_compression_runs/cache/`, copy it in to skip the 3-min cache-build. Otherwise the first call to `load_data_full` will rebuild from `videos/`.

In [ ]:
import shutil
REPO = Path('/content/comma_video_compression_challenge')
CKPT_DIR = REPO / 'autoresearch' / 'colab_run' / '3090_run'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Copy the e80 base model from Drive if present, else expect it already in repo
src = DRIVE_ROOT / 'h100_ckpts' / 'gen_3090.pt.e80.ckpt'
if not src.exists():
    src = DRIVE_ROOT / 'gen_3090.pt.e80.ckpt'
dst = CKPT_DIR / 'gen_3090.pt.e80.ckpt'
if src.exists() and not dst.exists():
    shutil.copy(src, dst); print(f'copied {src} -> {dst}')
elif dst.exists():
    print(f'have {dst} ({dst.stat().st_size//1024} KB)')
else:
    print(f'WARNING: e80 not found at {src} or {dst}. Upload it before running hypotheses.')

# Copy data cache if present
cache_src = DRIVE_ROOT / 'cache' / 'full_split_600all.pt'
cache_dst = REPO / 'autoresearch' / '_cache' / 'full_split_600all.pt'
cache_dst.parent.mkdir(parents=True, exist_ok=True)
if cache_src.exists() and not cache_dst.exists():
    print('copying data cache from Drive (~3.6GB, ~1 min)')
    shutil.copy(cache_src, cache_dst); print('done')

## 4. Common helpers (architecture variants + train loop + eval/ckpt manager)

Run this cell once. Defines:
- `eval_gen(gen)` → score dict
- `train_hypothesis(name, build_gen, *, epochs, eval_every, lr, batch_size)` — trains, evals every N epochs (and every 5 min wallclock), saves all ckpts + best to Drive.

In [ ]:
import sys, os, time, csv, math, copy
from pathlib import Path

REPO = Path('/content/comma_video_compression_challenge')
sys.path.insert(0, str(REPO / 'autoresearch'))
sys.path.insert(0, str(REPO))
os.chdir(str(REPO))

import torch, torch.nn as nn, torch.nn.functional as F
import einops

from train import (Generator, Trunk, Head1, Head2, Res, FiLMRes, DSConv,
                   QConv2d, QLinear, QEmb, load_data_full,
                   C1, C2, EMB_DIM, COND_DIM, HEAD_HIDDEN, DM,
                   GRAD_CLIP)
from prepare import (apply_fp4_to_model, fp4_round_trip, fake_quant_fp4_ste,
                     evaluate, gpu_cleanup, estimate_model_bytes,
                     MODEL_H, MODEL_W, OUT_H, OUT_W,
                     diff_round, pack_pair_yuv6, get_pose6,
                     load_segnet, load_posenet)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| GPU:', torch.cuda.get_device_name(0) if device.type=='cuda' else 'cpu')

BATCH_SIZE_DEFAULT = 8 if 'A100' in (torch.cuda.get_device_name(0) if device.type=='cuda' else '') else 4
if 'H100' in (torch.cuda.get_device_name(0) if device.type=='cuda' else ''):
    BATCH_SIZE_DEFAULT = 16
print('default batch size:', BATCH_SIZE_DEFAULT)

BASE_CKPT = REPO / 'autoresearch' / 'colab_run' / '3090_run' / 'gen_3090.pt.e80.ckpt'
BASE_SD   = torch.load(BASE_CKPT, map_location=device, weights_only=True)
print(f'base sd loaded: {len(BASE_SD)} tensors, {sum(v.numel() for v in BASE_SD.values())} elements')

# Single shared data load (3.6 GB).
DATA = load_data_full(device)
rgb, masks, poses = DATA['train_rgb'], DATA['train_masks'], DATA['train_poses']
segnet  = load_segnet(device)
posenet = load_posenet(device)
print(f'data ready: {rgb.shape[0]} pairs')

def eval_gen(gen):
    """Returns evaluate() dict. Note: evaluate calls apply_fp4_to_model in-place — caller must reload sd."""
    return evaluate(gen, DATA, device)

def model_byte_estimate(gen):
    return estimate_model_bytes(gen)

def train_hypothesis(name, build_gen, *, epochs=50, eval_every=5, lr=2e-4,
                     batch_size=None, ema_decay=0.9, time_budget_sec=None,
                     pose_weight=30.0, freeze_trunk_epochs=0, lr_min_factor=0.1):
    """
    name           : folder name under DRIVE_ROOT/prune/
    build_gen      : zero-arg callable that returns Generator-compatible nn.Module on `device`,
                     already initialized from BASE_SD as needed (mismatched layers initialized fresh).
    epochs         : max epochs (early stopping via time_budget_sec)
    eval_every     : eval every N epochs AND every 5 min wallclock (whichever first)
    freeze_trunk_epochs : if >0, trunk params are frozen for first N epochs (warmup new layers)
    """
    bs = batch_size or BATCH_SIZE_DEFAULT
    out_dir = PRUNE_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    log_csv = out_dir / 'log.csv'
    if not log_csv.exists():
        with open(log_csv, 'w', newline='') as f:
            csv.writer(f).writerow(['epoch','elapsed_sec','score','seg_term','pose_term','rate_term','model_bytes','ckpt'])

    gen = build_gen()
    n_p = sum(p.numel() for p in gen.parameters())
    bytes_est = estimate_model_bytes(gen)
    print(f'\n=== {name} ===  params={n_p}  est_bytes={bytes_est}  bs={bs}')

    # Initial eval BEFORE training (sanity)
    pre_res = eval_gen(gen)
    print(f'  pre-train eval: score={pre_res["score"]:.4f} seg={pre_res["seg_term"]:.4f} pose={pre_res["pose_term"]:.4f} rate={pre_res["rate_term"]:.4f}')
    # eval mutates weights via apply_fp4_to_model — rebuild fresh
    gen = build_gen()

    # Optimizer + QAT
    if hasattr(gen, 'set_qat'):
        gen.set_qat(True)
    if freeze_trunk_epochs > 0 and hasattr(gen, 'trunk'):
        for p in gen.trunk.parameters(): p.requires_grad = False
    opt = torch.optim.AdamW([p for p in gen.parameters() if p.requires_grad], lr=lr, betas=(0.9, 0.99))
    init_lr = lr

    ema_state = {k: v.detach().clone() for k, v in gen.state_dict().items()}

    t0 = time.time()
    best_score = float('inf')
    best_path = None
    last_eval_time = t0
    for epoch in range(epochs):
        if time_budget_sec and time.time() - t0 > time_budget_sec: break
        if freeze_trunk_epochs and epoch == freeze_trunk_epochs:
            for p in gen.trunk.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW(gen.parameters(), lr=lr, betas=(0.9, 0.99))
            print(f'  [unfreeze trunk @ epoch {epoch}]')
        # cosine LR
        progress = epoch / max(1, epochs)
        f_lr = lr_min_factor + (1 - lr_min_factor) * 0.5 * (1 + math.cos(math.pi * progress))
        for g in opt.param_groups: g['lr'] = init_lr * f_lr
        gen.train()
        g_perm = torch.Generator().manual_seed(42 + epoch)
        perm = torch.randperm(rgb.shape[0], generator=g_perm)
        ep_loss = 0.0; n_b = 0
        for s in range(0, rgb.shape[0], bs):
            idx = perm[s:s+bs]
            b_rgb  = rgb.index_select(0, idx).to(device, non_blocking=True)
            b_mask = masks.index_select(0, idx).to(device, non_blocking=True)
            b_pose = poses.index_select(0, idx).to(device, non_blocking=True)
            batch = einops.rearrange(b_rgb, 'b t h w c -> b t c h w').float()
            with torch.no_grad():
                r2 = F.interpolate(batch[:, 1], (MODEL_H, MODEL_W), mode='bilinear', align_corners=False)
                gt_logits = segnet(r2).float()
                gt_cls = gt_logits.argmax(1)
                gt_p = get_pose6(posenet, posenet.preprocess_input(batch)).float()
            opt.zero_grad(set_to_none=True)
            p1, p2 = gen(b_mask.long(), b_pose.float())
            f1u = F.interpolate(p1, (OUT_H, OUT_W), mode='bilinear', align_corners=False)
            f2u = F.interpolate(p2, (OUT_H, OUT_W), mode='bilinear', align_corners=False)
            f1d = F.interpolate(diff_round(f1u.clamp(0,255)), (MODEL_H, MODEL_W), mode='bilinear', align_corners=False)
            f2d = F.interpolate(diff_round(f2u.clamp(0,255)), (MODEL_H, MODEL_W), mode='bilinear', align_corners=False)
            pred_logits = segnet(f2d).float()
            ce = F.cross_entropy(pred_logits, gt_cls, reduction='none')
            with torch.no_grad():
                p_t = torch.exp(-ce.detach()).clamp_max(0.999)
                focal_w = (1.0 - p_t).pow(2.0)
            seg_loss = 100.0 * 25.0 * (focal_w * ce).mean()
            fp = get_pose6(posenet, pack_pair_yuv6(f1d, f2d).float()).float()
            pose_loss = pose_weight * F.mse_loss(fp, gt_p)
            loss = seg_loss + pose_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gen.parameters(), GRAD_CLIP)
            opt.step()
            with torch.no_grad():
                for k, v in gen.state_dict().items():
                    if v.dtype.is_floating_point:
                        ema_state[k].mul_(ema_decay).add_(v.detach(), alpha=1 - ema_decay)
                    else:
                        ema_state[k].copy_(v)
            ep_loss += loss.item(); n_b += 1
        elapsed = time.time() - t0

        # eval condition: every N epochs OR every 5 min
        do_eval = ((epoch + 1) % eval_every == 0) or (time.time() - last_eval_time > 300) or (epoch == epochs - 1)
        if do_eval:
            last_eval_time = time.time()
            # snapshot live, swap EMA in for eval
            cur = {k: v.detach().clone() for k, v in gen.state_dict().items()}
            gen.load_state_dict(ema_state)
            res = eval_gen(gen)
            score = res['score']
            ckpt_path = out_dir / f'{name}_e{epoch+1}_s{score:.4f}.ckpt'
            torch.save(ema_state, ckpt_path)
            with open(log_csv, 'a', newline='') as f:
                csv.writer(f).writerow([epoch+1, int(elapsed), f'{score:.6f}',
                                        f'{res["seg_term"]:.6f}', f'{res["pose_term"]:.6f}',
                                        f'{res["rate_term"]:.6f}', res['model_bytes'], ckpt_path.name])
            tag = ''
            if score < best_score:
                best_score = score
                best_path = out_dir / f'{name}_BEST.ckpt'
                torch.save(ema_state, best_path)
                tag = '   <-- NEW BEST'
            print(f'  ep {epoch+1:3d} | {int(elapsed):5d}s | loss {ep_loss/max(1,n_b):.3f} | '
                  f'score {score:.4f} seg {res["seg_term"]:.4f} pose {res["pose_term"]:.4f} '
                  f'bytes {res["model_bytes"]}{tag}', flush=True)
            # restore live weights
            gen.load_state_dict(cur)
        else:
            if (epoch+1) % 2 == 0:
                print(f'  ep {epoch+1:3d} | {int(elapsed):5d}s | loss {ep_loss/max(1,n_b):.3f}', flush=True)

    print(f'\n[{name}] DONE. best score={best_score:.4f}  ckpt={best_path}')
    del gen; gpu_cleanup()
    return best_score, best_path

## 5. Baseline sanity (no training)

In [ ]:
gen = Generator().to(device)
gen.load_state_dict(BASE_SD, strict=False)
res = eval_gen(gen)
print('e80 raw model (no sidecar):', res)
del gen; gpu_cleanup()
# Expect score ~0.2986. Sidecar pipeline takes us to 0.2921.

## H1 — pose_mlp → QLinear (FP4) + QAT  ⭐ START HERE

Replace the 3 `nn.Linear` layers in pose_mlp with `QLinear` (which uses internal QConv2d that gets FP4 byte treatment). Expected: 13.7 KB → 4.1 KB on pose_mlp = **~9.6 KB savings**.

Risk: pose conditioning is sensitive (FiLM gamma/beta). Local test showed naive FP4 round-trip blows pose_term 0.08→0.83. **QAT must recover.**

Strategy: freeze trunk for first 5 epochs while pose_mlp adapts, then unfreeze.

In [ ]:
class GeneratorPoseQ(Generator):
    """Generator with pose_mlp using QLinear (FP4-byte-counted) instead of nn.Linear."""
    def __init__(self):
        super().__init__()
        self.pose_mlp = nn.Sequential(
            QLinear(6, COND_DIM), nn.SiLU(),
            QLinear(COND_DIM, COND_DIM), nn.SiLU(),
            QLinear(COND_DIM, COND_DIM),
        )

def build_h1():
    g = GeneratorPoseQ().to(device)
    # Load all matching weights (everything except pose_mlp). Init pose_mlp from old weights via direct copy.
    sd = dict(BASE_SD)
    # Old: pose_mlp.0.weight (64, 6) fp32  -> New QLinear.conv.weight (64, 6, 1, 1)
    new_sd = {}
    for k, v in sd.items():
        if k.startswith('pose_mlp.'):
            # Old key: pose_mlp.{0,2,4}.{weight,bias}
            parts = k.split('.')
            new_k = f'pose_mlp.{parts[1]}.conv.{parts[2]}'
            if 'weight' in parts[2]:
                new_sd[new_k] = v.unsqueeze(-1).unsqueeze(-1)
            else:
                new_sd[new_k] = v
        else:
            new_sd[k] = v
    missing, unexpected = g.load_state_dict(new_sd, strict=False)
    print(f'  load: missing={len(missing)} unexpected={len(unexpected)}')
    if missing: print('   missing:', missing[:5])
    if unexpected: print('   unexpected:', unexpected[:5])
    return g

# Quick: train ~40 epochs (~30-45 min on A100), eval every 3 epochs, freeze trunk first 5
best_h1, ckpt_h1 = train_hypothesis(
    'H1_pose_qlinear', build_h1,
    epochs=60, eval_every=3, lr=3e-4,
    freeze_trunk_epochs=5, time_budget_sec=45*60,
)

## H2 — pose_mlp shrink: hidden 64→32 + QAT

Keep nn.Linear (no QAT-on-Linear risk) but make it smaller. 6→64, 64→32, 32→64. Saves param count: middle layer 4096→2048, last 4096→2048 — ~6 KB.

In [ ]:
class GeneratorPoseShrink(Generator):
    def __init__(self):
        super().__init__()
        H = 32  # was 64
        self.pose_mlp = nn.Sequential(
            nn.Linear(6, COND_DIM), nn.SiLU(),
            nn.Linear(COND_DIM, H), nn.SiLU(),
            nn.Linear(H, COND_DIM),
        )

def build_h2():
    g = GeneratorPoseShrink().to(device)
    sd = dict(BASE_SD)
    # First layer matches; layers .2 and .4 don't — init random, will train.
    g.load_state_dict({k: v for k, v in sd.items() if not k.startswith('pose_mlp.')}, strict=False)
    # Copy first pose layer
    g.pose_mlp[0].weight.data.copy_(sd['pose_mlp.0.weight'])
    g.pose_mlp[0].bias.data.copy_(sd['pose_mlp.0.bias'])
    return g

best_h2, ckpt_h2 = train_hypothesis(
    'H2_pose_shrink', build_h2,
    epochs=60, eval_every=3, lr=4e-4,
    freeze_trunk_epochs=8, time_budget_sec=45*60,
)

## H3 — SVD low-rank pose_mlp.2 + .4 + finetune

Replace each 64×64 nn.Linear with rank-r factorisation: `Linear(64, r)` → `Linear(r, 64)`. Init via SVD of the original weight (truncated to top-r singulars). For r=16: param count 64×64=4096 → 64×16+16×64=2048 (-50%). Save ~3-4 KB total.

Lower risk than QAT-on-Linear because we still use FP16.

In [ ]:
class LowRankLinear(nn.Module):
    def __init__(self, in_f, out_f, rank, bias=True):
        super().__init__()
        self.a = nn.Linear(in_f, rank, bias=False)
        self.b = nn.Linear(rank, out_f, bias=bias)
    def forward(self, x):
        return self.b(self.a(x))

RANK_H3 = 16

class GeneratorPoseLR(Generator):
    def __init__(self):
        super().__init__()
        self.pose_mlp = nn.Sequential(
            nn.Linear(6, COND_DIM), nn.SiLU(),
            LowRankLinear(COND_DIM, COND_DIM, RANK_H3, bias=True), nn.SiLU(),
            LowRankLinear(COND_DIM, COND_DIM, RANK_H3, bias=True),
        )

def svd_init(W, b, rank):
    """W: (out, in). Returns (A: rank,in), (B: out,rank), b unchanged."""
    U, S, Vh = torch.linalg.svd(W.float(), full_matrices=False)
    sqrt_s = S[:rank].sqrt()
    A = (sqrt_s.unsqueeze(1) * Vh[:rank])  # (rank, in)
    B = (U[:, :rank] * sqrt_s)             # (out, rank)
    return A, B

def build_h3():
    g = GeneratorPoseLR().to(device)
    sd = dict(BASE_SD)
    g.load_state_dict({k: v for k, v in sd.items() if not k.startswith('pose_mlp.')}, strict=False)
    g.pose_mlp[0].weight.data.copy_(sd['pose_mlp.0.weight'])
    g.pose_mlp[0].bias.data.copy_(sd['pose_mlp.0.bias'])
    for src_idx, mod_idx in [(2, 2), (4, 4)]:
        W = sd[f'pose_mlp.{src_idx}.weight']
        b = sd[f'pose_mlp.{src_idx}.bias']
        A, B = svd_init(W, b, RANK_H3)
        g.pose_mlp[mod_idx].a.weight.data.copy_(A.to(device))
        g.pose_mlp[mod_idx].b.weight.data.copy_(B.to(device))
        g.pose_mlp[mod_idx].b.bias.data.copy_(b.to(device))
    return g

best_h3, ckpt_h3 = train_hypothesis(
    'H3_pose_svd_r16', build_h3,
    epochs=40, eval_every=3, lr=2e-4,
    freeze_trunk_epochs=4, time_budget_sec=30*60,
)

## H4 — Drop second FiLMRes (h1.r2) + QAT

Head1 has TWO FiLMRes blocks. Hypothesis: one is enough. Saves ~7 KB (DSConv c1 + dw2 + pw2 + film conv). Higher risk — may need more retraining.

In [ ]:
class Head1Single(nn.Module):
    def __init__(self):
        super().__init__()
        self.r1 = FiLMRes(C1, COND_DIM)
        self.out = QConv2d(C1, 3, 1, quantize_weight=False)
    def forward(self, f, c):
        return torch.sigmoid(self.out(self.r1(f, c))) * 255.0

class GeneratorH1Single(Generator):
    def __init__(self):
        super().__init__()
        self.h1 = Head1Single()

def build_h4():
    g = GeneratorH1Single().to(device)
    g.load_state_dict({k: v for k, v in BASE_SD.items() if not k.startswith('h1.r2.')}, strict=False)
    return g

best_h4, ckpt_h4 = train_hypothesis(
    'H4_drop_h1r2', build_h4,
    epochs=70, eval_every=3, lr=3e-4,
    freeze_trunk_epochs=10, time_budget_sec=60*60,
)

## H5 — Trunk C2 (bottleneck width) 64 → 48 + QAT

Reduce the bottleneck channel count in trunk.down/d1/up. Affects 4 layers. Saves ~3 KB.

In [ ]:
C2_NEW = 48

class TrunkC2Small(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = QEmb(5, EMB_DIM, quantize_weight=False)
        self.stem = DSConv(EMB_DIM + 2, C1)
        self.s1 = Res(C1)
        self.down = DSConv(C1, C2_NEW, s=2)
        self.d1 = Res(C2_NEW)
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            DSConv(C2_NEW, C1),
        )
        self.fuse = DSConv(C1*2, C1)
        self.f1 = Res(C1)
    def forward(self, mask, co):
        e = F.interpolate(self.emb(mask.long()).permute(0,3,1,2), co.shape[-2:], mode='bilinear', align_corners=False)
        s = self.s1(self.stem(torch.cat([e, co], 1)))
        z = self.up(self.d1(self.down(s)))
        return self.f1(self.fuse(torch.cat([z, s], 1)))

class GeneratorC2Small(Generator):
    def __init__(self):
        super().__init__()
        self.trunk = TrunkC2Small()

def build_h5():
    g = GeneratorC2Small().to(device)
    # Load matching layers; init bottleneck width-mismatched layers fresh
    safe = {}
    for k, v in BASE_SD.items():
        if k.startswith('trunk.down.') or k.startswith('trunk.d1.') or k.startswith('trunk.up.'):
            continue  # shape mismatch, init fresh
        safe[k] = v
    missing, unexpected = g.load_state_dict(safe, strict=False)
    print(f'  load: missing={len(missing)} unexpected={len(unexpected)}')
    return g

best_h5, ckpt_h5 = train_hypothesis(
    'H5_C2_48', build_h5,
    epochs=80, eval_every=4, lr=4e-4,
    freeze_trunk_epochs=0, time_budget_sec=60*60,
)

## H6 — HEAD_HIDDEN 52 → 32 + QAT

Head2 internal width. Saves ~1 KB. Low risk — Head2 is reconstruction, less critical than pose path.

In [ ]:
HH_NEW = 32

class Head2Small(nn.Module):
    def __init__(self):
        super().__init__()
        self.r1 = Res(C1)
        self.pre = DSConv(C1, HH_NEW)
        self.out = QConv2d(HH_NEW, 3, 1, quantize_weight=False)
    def forward(self, f):
        return torch.sigmoid(self.out(self.pre(self.r1(f)))) * 255.0

class GeneratorH2Small(Generator):
    def __init__(self):
        super().__init__()
        self.h2 = Head2Small()

def build_h6():
    g = GeneratorH2Small().to(device)
    safe = {k: v for k, v in BASE_SD.items() if not k.startswith('h2.pre.') and not k.startswith('h2.out.')}
    g.load_state_dict(safe, strict=False)
    return g

best_h6, ckpt_h6 = train_hypothesis(
    'H6_head_hidden_32', build_h6,
    epochs=40, eval_every=3, lr=4e-4,
    freeze_trunk_epochs=4, time_budget_sec=30*60,
)

## H7 — Drop h2.r1 (Res block in Head2)

Head2 has Res → DSConv → 1×1 conv. Drop the Res. Saves ~3 KB. Will it hurt seg? seg_term is at noise floor 0.000291, so we have margin.

In [ ]:
class Head2NoRes(nn.Module):
    def __init__(self):
        super().__init__()
        self.pre = DSConv(C1, HEAD_HIDDEN)
        self.out = QConv2d(HEAD_HIDDEN, 3, 1, quantize_weight=False)
    def forward(self, f):
        return torch.sigmoid(self.out(self.pre(f))) * 255.0

class GeneratorH2NoRes(Generator):
    def __init__(self):
        super().__init__()
        self.h2 = Head2NoRes()

def build_h7():
    g = GeneratorH2NoRes().to(device)
    safe = {k: v for k, v in BASE_SD.items() if not k.startswith('h2.r1.')}
    g.load_state_dict(safe, strict=False)
    return g

best_h7, ckpt_h7 = train_hypothesis(
    'H7_drop_h2r1', build_h7,
    epochs=50, eval_every=3, lr=3e-4,
    freeze_trunk_epochs=5, time_budget_sec=45*60,
)

## H8 — 🎯 COMBINED: H1 (pose QLinear) + H4 (drop h1.r2) 

If both work alone, combine them: ~16 KB total savings = score → ~0.281! Higher risk — needs longer retrain. Run AFTER H1 and H4 individually validate.

In [ ]:
class GeneratorCombo(Generator):
    def __init__(self):
        super().__init__()
        self.pose_mlp = nn.Sequential(
            QLinear(6, COND_DIM), nn.SiLU(),
            QLinear(COND_DIM, COND_DIM), nn.SiLU(),
            QLinear(COND_DIM, COND_DIM),
        )
        self.h1 = Head1Single()  # from H4

def build_h8():
    g = GeneratorCombo().to(device)
    sd = dict(BASE_SD)
    new_sd = {}
    for k, v in sd.items():
        if k.startswith('pose_mlp.'):
            parts = k.split('.')
            new_k = f'pose_mlp.{parts[1]}.conv.{parts[2]}'
            new_sd[new_k] = v.unsqueeze(-1).unsqueeze(-1) if 'weight' in parts[2] else v
        elif k.startswith('h1.r2.'):
            continue
        else:
            new_sd[k] = v
    missing, unexpected = g.load_state_dict(new_sd, strict=False)
    print(f'  load: missing={len(missing)} unexpected={len(unexpected)}')
    return g

best_h8, ckpt_h8 = train_hypothesis(
    'H8_combo_poseQ_dropH1r2', build_h8,
    epochs=90, eval_every=3, lr=3e-4,
    freeze_trunk_epochs=10, time_budget_sec=90*60,
)

## H9 — Distillation: smaller student (C1: 56→44) from e80 teacher

Most ambitious. Train a fresh smaller model with C1=44 (instead of 56), distill from e80 outputs (frame-level KL on f1, MSE on f2). Saves ~6-8 KB if it works.

In [ ]:
C1_NEW = 44
HH_DIST = 36

class TrunkSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = QEmb(5, EMB_DIM, quantize_weight=False)
        self.stem = DSConv(EMB_DIM + 2, C1_NEW)
        self.s1 = Res(C1_NEW)
        self.down = DSConv(C1_NEW, C2, s=2)
        self.d1 = Res(C2)
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            DSConv(C2, C1_NEW),
        )
        self.fuse = DSConv(C1_NEW*2, C1_NEW)
        self.f1 = Res(C1_NEW)
    def forward(self, mask, co):
        e = F.interpolate(self.emb(mask.long()).permute(0,3,1,2), co.shape[-2:], mode='bilinear', align_corners=False)
        s = self.s1(self.stem(torch.cat([e, co], 1)))
        z = self.up(self.d1(self.down(s)))
        return self.f1(self.fuse(torch.cat([z, s], 1)))

class Head1Small(nn.Module):
    def __init__(self):
        super().__init__()
        self.r1 = FiLMRes(C1_NEW, COND_DIM)
        self.r2 = FiLMRes(C1_NEW, COND_DIM)
        self.out = QConv2d(C1_NEW, 3, 1, quantize_weight=False)
    def forward(self, f, c):
        return torch.sigmoid(self.out(self.r2(self.r1(f, c), c))) * 255.0

class Head2SmallDist(nn.Module):
    def __init__(self):
        super().__init__()
        self.r1 = Res(C1_NEW)
        self.pre = DSConv(C1_NEW, HH_DIST)
        self.out = QConv2d(HH_DIST, 3, 1, quantize_weight=False)
    def forward(self, f):
        return torch.sigmoid(self.out(self.pre(self.r1(f)))) * 255.0

class GeneratorSmall(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = TrunkSmall()
        self.pose_mlp = nn.Sequential(
            QLinear(6, COND_DIM), nn.SiLU(),
            QLinear(COND_DIM, COND_DIM), nn.SiLU(),
            QLinear(COND_DIM, COND_DIM),
        )
        self.h1 = Head1Small()
        self.h2 = Head2SmallDist()
    def set_qat(self, on):
        for m in self.modules():
            if isinstance(m, (QConv2d, QEmb)): m.qat = on
    def forward(self, mask, pose):
        from train import coords as _coords
        co = _coords(mask.shape[0], MODEL_H, MODEL_W, mask.device)
        feat = self.trunk(mask, co)
        cond = self.pose_mlp(pose)
        return self.h1(feat, cond), self.h2(feat)

def build_h9():
    g = GeneratorSmall().to(device)
    return g  # full random init; distillation will train

# H9 needs distillation loss — use teacher outputs as soft targets.
# Re-uses train_hypothesis for now (uses real seg/pose loss). For pure distill, see optional cell below.
best_h9, ckpt_h9 = train_hypothesis(
    'H9_distill_C1_44', build_h9,
    epochs=120, eval_every=5, lr=5e-4,
    time_budget_sec=90*60, ema_decay=0.95,
)

## Summary cell — list all checkpoints + scores from Drive

Run anytime to see leaderboard.

In [ ]:
import csv as _csv
rows = []
for log in sorted(PRUNE_DIR.glob('*/log.csv')):
    name = log.parent.name
    with open(log) as f:
        for r in _csv.DictReader(f):
            rows.append({'name': name, **r})

if not rows:
    print('no logs yet')
else:
    rows.sort(key=lambda r: float(r['score']))
    print(f"{'rank':<4} {'name':<28} {'epoch':>5} {'score':>8} {'seg':>8} {'pose':>8} {'rate':>8} {'bytes':>7}")
    for i, r in enumerate(rows[:30], 1):
        print(f"{i:<4} {r['name']:<28} {r['epoch']:>5} {float(r['score']):>8.4f} "
              f"{float(r['seg_term']):>8.4f} {float(r['pose_term']):>8.4f} "
              f"{float(r['rate_term']):>8.4f} {r['model_bytes']:>7}")

## (Optional) Pure-distillation training cell for H9-style runs

If H9 with seg/pose loss doesn't converge, try this: distill student outputs to match teacher's f1/f2 directly (no segnet/posenet in the loop, much faster per step).

In [ ]:
def train_distill(name, build_student, *, epochs=60, eval_every=3, lr=5e-4,
                  batch_size=None, time_budget_sec=60*60):
    bs = batch_size or BATCH_SIZE_DEFAULT
    out_dir = PRUNE_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    log_csv = out_dir / 'log.csv'
    if not log_csv.exists():
        with open(log_csv, 'w', newline='') as f:
            csv.writer(f).writerow(['epoch','elapsed_sec','score','seg_term','pose_term','rate_term','model_bytes','ckpt'])

    teacher = Generator().to(device)
    teacher.load_state_dict(BASE_SD, strict=False)
    apply_fp4_to_model(teacher)
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False

    student = build_student()
    if hasattr(student, 'set_qat'): student.set_qat(True)
    opt = torch.optim.AdamW(student.parameters(), lr=lr, betas=(0.9, 0.99))
    ema_state = {k: v.detach().clone() for k, v in student.state_dict().items()}
    t0 = time.time(); best = float('inf'); best_path = None; last_eval_time = t0
    for epoch in range(epochs):
        if time_budget_sec and time.time() - t0 > time_budget_sec: break
        for g in opt.param_groups: g['lr'] = lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * epoch / max(1, epochs))))
        student.train()
        g_perm = torch.Generator().manual_seed(42 + epoch)
        perm = torch.randperm(rgb.shape[0], generator=g_perm)
        ep_loss = 0.0; n_b = 0
        for s in range(0, rgb.shape[0], bs):
            idx = perm[s:s+bs]
            b_mask = masks.index_select(0, idx).to(device, non_blocking=True)
            b_pose = poses.index_select(0, idx).to(device, non_blocking=True)
            with torch.no_grad():
                t1, t2 = teacher(b_mask.long(), b_pose.float())
            opt.zero_grad(set_to_none=True)
            s1, s2 = student(b_mask.long(), b_pose.float())
            loss = F.l1_loss(s1, t1) + F.l1_loss(s2, t2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
            opt.step()
            with torch.no_grad():
                for k, v in student.state_dict().items():
                    if v.dtype.is_floating_point:
                        ema_state[k].mul_(0.9).add_(v.detach(), alpha=0.1)
                    else:
                        ema_state[k].copy_(v)
            ep_loss += loss.item(); n_b += 1
        elapsed = time.time() - t0
        if (epoch+1) % eval_every == 0 or time.time() - last_eval_time > 300 or epoch == epochs - 1:
            last_eval_time = time.time()
            cur = {k: v.detach().clone() for k, v in student.state_dict().items()}
            student.load_state_dict(ema_state)
            res = eval_gen(student)
            score = res['score']
            ckpt_path = out_dir / f'{name}_e{epoch+1}_s{score:.4f}.ckpt'
            torch.save(ema_state, ckpt_path)
            tag = ''
            if score < best:
                best = score
                best_path = out_dir / f'{name}_BEST.ckpt'
                torch.save(ema_state, best_path)
                tag = '   <-- NEW BEST'
            with open(log_csv, 'a', newline='') as f:
                csv.writer(f).writerow([epoch+1, int(elapsed), f'{score:.6f}', f'{res["seg_term"]:.6f}',
                                        f'{res["pose_term"]:.6f}', f'{res["rate_term"]:.6f}', res['model_bytes'], ckpt_path.name])
            print(f'  ep {epoch+1:3d} | {int(elapsed):5d}s | distill {ep_loss/max(1,n_b):.3f} | '
                  f'score {score:.4f} bytes {res["model_bytes"]}{tag}', flush=True)
            student.load_state_dict(cur)
    print(f'\n[{name}] DONE. best score={best:.4f}  ckpt={best_path}')
    del student, teacher; gpu_cleanup()
    return best, best_path

# Example: distill smaller student
# best, ck = train_distill('H9_distill', build_h9, epochs=120, eval_every=5, time_budget_sec=90*60)